# 2-clean&filter

In [38]:
import pandas as pd
import re

In [39]:
df = pd.read_csv(
    "../data/interim/extract_15_16_concat.csv",
    low_memory=False,
    dtype={
        "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)
df.shape

(1128128, 26)

In [40]:
# Ne garder que le code style NORMAL
df = df[df["code_style"] == "NORMAL"]

# Exclure les prises de parole de "Mme la présidente" et "M. le président"
# Role_debat n'est pas bien identifié, utiliser Nom_orateur
df = df[~df["nom_orateur"].str.strip().isin(["M. le président", "Mme la présidente"])]

# Garder une trace de la longueur des interventions brutes
df["len_dirtytext"] = df["texte"].str.len()

# TODO: virer même le ID_orateur si tout est bien stable
# TODO: attendre un peu, visiblement moyen de récupérer ainsi des id-acteur manquants
# Stabiliser le ID_orateur pour etre au format AN (pour matcher données)
# marche car pandas propage les NaN quand bien reconnu comme objet
# donc l'importance de str au chargement (et de pas forcer en str après ?)
df["id_orateur"] = "PA" + df["id_orateur"]

# même si en fait au final on va utiliser id_acteur

# Changer les missing values pour non_précisé (majoritaire) dans Code_parole
df["code_parole"] = df["code_parole"].fillna("non_précisé")

df.shape

(683680, 27)

In [41]:
# TODO: certains id_acteurs manquants ont des bons id_orateur
# vérifier si on peut les récupérer et les renvoyer à  la place de rien pour ces cas limite
# certains PA0 par contre, bizarres mais pourra rien pour eux.

In [42]:
df["id_orateur"].isna().sum()  # 0

44952

In [43]:
df["id_acteur"].isna().sum()  # 0

308

In [44]:
# aperçu des répartitions
df.groupby("code_parole", dropna=False)["len_dirtytext"].describe()

# TODO: voir à la toute fin si besoin de modifier les codes pris en compte avis_ ?

,count,mean,std,min,25%,50%,75%,max
code_parole,,,,,,,,
(null),1.0,186.000000,NaN,186.0,186.00,186.0,186.00,186.0
AVIS_COM_1_10,1.0,1278.000000,NaN,1278.0,1278.00,1278.0,1278.00,1278.0
AVIS_COM_1_20,45107.0,450.390582,482.536522,3.0,118.50,315.0,618.00,8179.0
AVIS_GVT_1_20,38454.0,486.647371,681.527170,4.0,17.00,240.0,685.00,11340.0
PAROLE_1_1,21.0,556.761905,1080.603346,30.0,101.00,241.0,529.00,5088.0
PAROLE_1_2,247778.0,974.519219,1294.330875,3.0,245.00,547.0,1207.00,68092.0
PRESIDE_DISCOURS_1_10,2.0,95.500000,92.630988,30.0,62.75,95.5,128.25,161.0
Raccroche_apres_inter,9.0,1638.666667,1450.025431,60.0,154.00,1340.0,2316.00,3939.0
non_précisé,352287.0,282.342874,625.719777,1.0,17.00,37.0,251.00,25609.0


In [45]:
# TODO: regrouper les interventions interrompues ?

**?????????aviser pour regrouper les interventions interrompues ?????????**

## Match députés

### Match infos générales (historique)

In [46]:
df_deputes = pd.read_csv("../data/raw/id-dep/deputes-historique(datan-datagouv).csv")
# suppression des colonnes non utiles qui introduisent soucis parsing
df_deputes = df_deputes.drop(columns=["mail", "twitter", "facebook", "website"])


In [47]:
print("shape avant fusion:", df.shape)

assert df_deputes["id"].is_unique, "ids du df_deputes non uniques !"

# Merge et virer la col id pour éviter doublon
df = df.merge(
    df_deputes,
    left_on="id_acteur",
    right_on="id",
    how="left",
    suffixes=("", "_dep"),
    validate="many_to_one",  # check if merge keys are unique in right dataset
).drop(columns=["id"])  # supprimer la colonne id du df_deputes

print("shape après fusion:", df.shape)

shape avant fusion: (683680, 27)
shape après fusion: (683680, 49)


### Match temporel des affiliations

### Recodage des dénominations
(Mais attention : ici choix de recoder avec nom des partis, alors que les groupes parlementaires sont + larges que les partis et peuvent servir à acceuillir des NI d'étiquettes diverses comme le groupe ECO qui acceuillent les députés de l'Après, Génération.s et Ruffin)

In [48]:
# recodage des grandes dénominations des groupes
# (moins sensible aux évolutions marginales de dénomination)

df_affiliation = pd.read_csv(
    "../data/raw/id-dep/datan_affiliations.csv", encoding="latin1", sep=";"
)  # format dégueu

# Recoder les partis pour stabilité temporelle des noms
# TODO: trancher pour dernires regroupements (ou garder pour ultérieur sur les blocs électoraux)
# Vérifier
recodage = {
    "RE": "REN",
    "LAREM": "REN",
    "MODEM": "DEM",
    "SOC": "SOC-A",
    "NG": "SOC-A",
    "LFI-NUPES": "LFI",
    "FI": "LFI",
    "UDI-AGIR": "UDI",
    "UDI-A-I": "UDI",
    "LC": "UDI",
    "UDI_I": "UDI",
    "UDI-I": "UDI",
    "ECOLO": "ECO",
    "GDR-NUPES": "GDR",
    "LT": "LIOT",
    # Garde pour trace mais pas nécessaire car pas de changement
    # "LIOT": "LIOT",
    # "LR": "LR",
    # "RN": "RN",
    # "MODEM": "MODEM",
    # "LFI": "LFI",
    # "HOR": "HOR",
    # "DEM": "DEM",
}

df_affiliation["libelleAbrev"] = df_affiliation["libelleAbrev"].astype(str).str.strip()
df_affiliation["parti_recod"] = df_affiliation["libelleAbrev"].replace(recodage)


In [49]:
# Plutôt qu'un merge foireux parti sur un lookup ligne‑à‑ligne
# (= pb des orateurs non députés qui étaient pas présents, etc.)
# Le fichier est suffisamment réduit pour que le surplus de calcul soit pas un pb


# préparation des dates
df["dateSeance_ts"] = pd.to_datetime(
    df["dateSeance"], format="%Y%m%d%H%M%S%f", errors="raise"
)
df_affiliation["dateDebut"] = pd.to_datetime(
    df_affiliation["dateDebut"], errors="raise"
)
df_affiliation["dateFin"] = pd.to_datetime(df_affiliation["dateFin"], errors="raise")
# # aviser si jamais besoin traiter affiliations en cours
# df_affiliation["dateFin"] = df_affiliation["dateFin"].fillna(pd.Timestamp("2100-01-01"))

# indexer par mpId pour lookup rapide
aff_by_mp = {
    mp: g[["dateDebut", "dateFin", "parti_recod"]].to_dict("records")
    for mp, g in df_affiliation.groupby("mpId")
}


def get_parti_for_row(row):
    mp = row.get("id_acteur")  # correspond au mpId
    # gérer le cas des orateurs non députés ou autre type intervention
    if pd.isna(mp) or mp not in aff_by_mp:
        return None
    # récupérer le ts de l'intervention
    ts = row.get("dateSeance_ts")
    if pd.isna(ts):
        return None
    # retourner l'affiliation qui colle à la date d'intervention
    for rec in aff_by_mp[mp]:
        # attention : .normalize() pour ignorer l'heure car sinon hors des bornes de fin
        if rec["dateDebut"] <= ts.normalize() <= rec["dateFin"]:
            return rec["parti_recod"]
    return None


# appliquer
df["parti_affiliation"] = df.apply(get_parti_for_row, axis=1)

# Pas parfait mais pour avoir une idée :
print(
    "affectés :",
    df["parti_affiliation"].notna().sum(),
    # Eux on sait pas (pas députés, autre code parole intervention, etc.)
    "| non affectés :",
    df["parti_affiliation"].isna().sum(),
)


affectés : 563082 | non affectés : 120598


In [50]:
# Recodage des RN de la XVe législature au bloc RN
# = initialement en NI car pas assez nombreux pour former un groupe

liste_NI_RN = [
    "PA720822",
    "PA720668",
    "PA720468",
    "PA720614",
    "PA719436",
    "PA720802",
    "PA719608",
    "PA720606",
    "PA606212",
    "PA720798",
]

df.loc[df["id_acteur"].isin(liste_NI_RN), "parti_affiliation"] = "RN"


In [51]:
# TODO: envisager de forcer le renvoi de la derniere affiliation connue de df_deputes ?
# Aviser pour des matchs plus précis (cas limites, etc.) sur la base des repérages de matthias.

In [52]:
df["parti_affiliation"].value_counts()

parti_affiliation
REN       131692
LR        131124
LFI        78970
SOC-A      43423
GDR        42546
DEM        39360
RN         29357
UDI        18681
LIOT       18387
ECO        13970
NI          6404
HOR         5043
AGIR-E      3176
EDS          949
Name: count, dtype: int64

In [53]:
df["groupeAbrev"].value_counts()

groupeAbrev
EPR          73731
DR           68855
LFI-NFP      52659
LAREM        52162
DEM          49851
RE           48758
LR           48150
SOC          33488
NI           27618
GDR          24686
ECOS         24612
RN           23618
LIOT         20948
GDR-NUPES    17449
HOR          15020
LFI-NUPES    10119
FI            6734
UDI_I         6585
LT            6287
SOC-A         5984
AGIR-E        4767
LES-REP       3306
UDR           1734
UMP            410
UDI-AGIR       179
ECOLO          144
NG             134
MODEM           43
Name: count, dtype: int64

In [54]:
def nettoyer_nom(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour avoir les bons apostrophes
    texte = texte.replace("’", "'")
    return texte


df["nom_orateur_clean"] = df["nom_orateur"].apply(nettoyer_nom)

In [55]:
# TODO: harmoniser mieux les NOM ORATEURS ( "", ministre, etc."")
# TODO: faire un fuzzyfuzz moche au besoin ?
# TODO: Ou plutôt renvoyer le nom le plus présent par id_acteur ?

In [56]:
# TODO: regrouper les interventions interrompues ?

## Export

In [57]:
# Export du csv nettoyé
df.to_csv("../data/interim/data_cleaning.csv", index=False)

# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

# import csv  # pour utiliser csv.QUOTE_ALL et résoudre le soucis d'écart.
# df.to_csv(
#     "../data/interim/data_cleaning.csv",
#     index=False,
#     quoting=csv.QUOTE_ALL,  # permet de résoudre le soucis
# )

In [58]:
# # verif ecriture/lecture ok
# print("df shape:", df.shape)

# df_test = pd.read_csv("../data/interim/data_cleaning.csv", low_memory=False)

# print("df_test shape (après export import): ", df_test.shape)

## Exploration

In [59]:
# TODO: vérifier les modifs RN :
# On passe de
# RN          21684
# à
# RN         28977

# Et les NI de
# NI          12966
# à
# NI          5673

# 7293

# TODO: possible de checker contre groupeAbrev

In [60]:
# Sélectionner les députés RN selon l'affiliation
df_rn = df[df["parti_affiliation"] == "RN"]

# Comparer la colonne 'groupeAbrev' pour ces députés
# utiliser ou non drop_duplicates pour éviter les doublons selon but
comparison = df_rn[
    ["id_acteur", "nom_orateur", "parti_affiliation", "groupeAbrev"]
]  # .drop_duplicates()

# Afficher les différentes valeurs de groupeAbrev pour les RN
print(comparison["groupeAbrev"].value_counts())

comparison

groupeAbrev
RN    23391
NI     5966
Name: count, dtype: int64


,id_acteur,nom_orateur,parti_affiliation,groupeAbrev
333,PA719608,Mme Emmanuelle Ménard,RN,NI
371,PA719608,Mme Emmanuelle Ménard,RN,NI
373,PA719608,Mme Emmanuelle Ménard,RN,NI
405,PA719608,Mme Emmanuelle Ménard,RN,NI
510,PA719608,Mme Emmanuelle Ménard,RN,NI
...,...,...,...,...
683608,PA795836,M. Frédéric Cabrolier,RN,RN
683617,PA795836,M. Frédéric Cabrolier (RN),RN,RN
683619,PA795836,M. Frédéric Cabrolier,RN,RN
683649,PA795836,M. Frédéric Cabrolier (RN),RN,RN


In [61]:
# explorer les affiliation manquantes pour identifier les cas limites
# Notamment regarder ceux qui n'ont pas d'affiliation mais bien un groupeAbrev
# Et aviser si on veut forcer l'affiliation à la derniere connue dans df_deputes
# En réalité sans doute des membres du gouvernement, donc toujours le même souci
# de décision à prendre selon l'usage qu'on veut faire des données
# -> ok avec Vincent, ça se défend sans pb.

In [62]:
df_missing_affil = df[df["groupeAbrev"].notna() & df["parti_affiliation"].isna()]

In [63]:
df_missing_affil.shape

(64949, 52)

In [65]:
df_missing_affil[["id_acteur", "nom_orateur", "groupeAbrev", "dateSeance_ts"]]

,id_acteur,nom_orateur,groupeAbrev,dateSeance_ts
334,PA331481,M. Bruno Le Maire,LAREM,2019-03-14 15:00:00
336,PA331481,M. Bruno Le Maire,LAREM,2019-03-14 15:00:00
339,PA331481,M. Bruno Le Maire,LAREM,2019-03-14 15:00:00
345,PA331481,M. Bruno Le Maire,LAREM,2019-03-14 15:00:00
347,PA331481,M. Bruno Le Maire,LAREM,2019-03-14 15:00:00
...,...,...,...,...
683670,PA721498,M. Stanislas Guerini,RE,2023-02-27 16:00:00
683672,PA721498,M. Stanislas Guerini,RE,2023-02-27 16:00:00
683675,PA721498,M. Stanislas Guerini,RE,2023-02-27 16:00:00
683677,PA721498,M. Stanislas Guerini,RE,2023-02-27 16:00:00


In [66]:
df_missing_affil["id_acteur"].nunique()

60

In [67]:
df_missing_affil[
    "nom_orateur"
].unique()  # différent de nb id_acteur unique car pb d'harmonisation noms

array(['M. Bruno Le Maire', 'M. Edouard Philippe', 'M. Olivier Dussopt',
       'M. Benjamin Griveaux', 'M. Stéphane Travert', 'Mme Brune Poirson',
       'M. Christophe Castaner', 'M. Édouard Philippe',
       'M. Jean-Yves Le Drian', 'Mme Bérangère Abba',
       'Mme Barbara Pompili', 'Mme Élisabeth Borne',
       'M. Jean-Baptiste Djebbari', 'Mme Nathalie Elimas',
       'Mme Brigitte Bourguignon', 'Mme Brigitte Klinkert',
       'Mme Annick Girardin', 'M. Mounir Mahjoubi', 'M. Gérald Darmanin',
       'M. Olivier Véran', 'Mme Agnès Pannier-Runacher',
       'M. Marc Fesneau', 'M. Clément Beaune', 'Mme Sarah El Haïry',
       'Mme Geneviève Darrieussecq', 'M. Franck Riester',
       'Mme Olivia Grégoire', 'M. Laurent Pietraszewski',
       'M. François de Rugy', 'Mme Amélie de Montchalin', 'Mme Nadia Hai',
       'Mme Roselyne Bachelot', 'Mme Christelle Dubos',
       'M. Gabriel Attal', 'M. Adrien Taquet', 'M. Éric Poulliat',
       'Mme Olivia Gregoire', 'Mme Barbara Pompili,', 'M

In [68]:
df_missing_affil["id_acteur"].value_counts()[
    :50
]  # différent de nb id_acteur unique car pb d'harmonisation noms

id_acteur
PA607846    8432
PA330357    5831
PA331481    4421
PA717161    4139
PA759832    4043
PA642788    3613
PA722190    3251
PA345619    2326
PA719938    2290
PA722086    2062
PA721134    1926
PA609520    1306
PA608083    1252
PA720512    1245
PA607395    1212
PA721872    1168
PA721764    1104
PA332747    1049
PA605131    1049
PA267797     980
PA1872       910
PA721836     814
PA335758     786
PA774109     723
PA719660     697
PA793788     580
PA793940     576
PA337633     563
PA719914     491
PA720002     443
PA332        410
PA722046     347
PA720242     339
PA793278     334
PA722236     329
PA267780     298
PA721670     284
PA721560     269
PA826635     248
PA719624     239
PA720170     235
PA795920     206
PA643184     205
PA795350     202
PA267336     192
PA331582     190
PA720924     187
PA719218     185
PA721498     183
PA719186     138
Name: count, dtype: int64

In [69]:
df_missing_affil["nom_orateur"].value_counts()[
    :50
]  # différent de nb id_acteur unique car pb d'harmonisation noms

nom_orateur
M. Gérald Darmanin              8431
M. Olivier Dussopt              5831
M. Bruno Le Maire               4421
Mme Élisabeth Borne             4139
Mme Agnès Pannier-Runacher      4043
M. Olivier Véran                3613
M. Gabriel Attal                3247
M. Marc Fesneau                 2287
M. Adrien Taquet                2065
M. Roland Lescure               1926
Mme Barbara Pompili             1305
M. Édouard Philippe             1301
Mme Brigitte Bourguignon        1252
M. Laurent Pietraszewski        1245
M. Stéphane Travert             1212
Mme Brune Poirson               1168
M. Christophe Castaner          1049
M. François de Rugy             1049
M. Edouard Philippe             1025
Mme Catherine Vautrin            980
Mme Olivia Grégoire              945
M. Jean-Yves Le Drian            910
M. Jean-Noël Barrot              814
M. Franck Riester                786
M. Clément Beaune                723
Mme Christelle Dubos             697
Mme Dominique Faure       

In [70]:
df_missing_affil["nom_orateur_clean"].value_counts()[:50]

nom_orateur_clean
M. Gérald Darmanin              8431
M. Olivier Dussopt              5831
M. Bruno Le Maire               4421
Mme Élisabeth Borne             4139
Mme Agnès Pannier-Runacher      4043
M. Olivier Véran                3613
M. Gabriel Attal                3247
M. Marc Fesneau                 2287
M. Adrien Taquet                2065
M. Roland Lescure               1926
Mme Barbara Pompili             1305
M. Édouard Philippe             1301
Mme Brigitte Bourguignon        1252
M. Laurent Pietraszewski        1245
M. Stéphane Travert             1212
Mme Brune Poirson               1168
M. Christophe Castaner          1049
M. François de Rugy             1049
M. Edouard Philippe             1025
Mme Catherine Vautrin            980
Mme Olivia Grégoire              946
M. Jean-Yves Le Drian            910
M. Jean-Noël Barrot              814
M. Franck Riester                786
M. Clément Beaune                723
Mme Christelle Dubos             697
Mme Dominique Faure 

In [71]:
# TODO: affiner affiliation

# Avoir plusieurs variables
# une d'info gouv vs députés

# pour affiliation
# une des députés (le reste en missing) = ce que l'on a
# une des députes + membres gouv = les afficher en tant que tel comme "groupe"ArithmeticError
# une des députés + ancienne affiliation des membres gouv = compléter les missing
# TODO: matthias : check les cas particuliers.


In [72]:
df["id_acteur"].isna().sum()  # 0

308

In [73]:
chelou = df[df["id_acteur"].isna()]

In [74]:
chelou

,UID,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,experienceDepute,scoreParticipation,scoreParticipationSpecialite,scoreLoyaute,scoreMajorite,active,dateMaj,dateSeance_ts,parti_affiliation,nom_orateur_clean
6073,CRSANR5L15S2017E1N004,NaN,NaN,20170706150000000,jeudi 06 juillet 2017,2,4,AN,15,Première session extraordinaire 2017,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-07-06 15:00:00,None,NaN
11867,CRSANR5L15S2019O1N018,NaN,NaN,20181015213000000,lundi 15 octobre 2018,2,18,AN,15,Session ordinaire 2018-2019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-10-15 21:30:00,None,Mme Christine Pires Beaune
11879,CRSANR5L15S2019O1N018,NaN,NaN,20181015213000000,lundi 15 octobre 2018,2,18,AN,15,Session ordinaire 2018-2019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-10-15 21:30:00,None,M. Jean-Louis Bricout
12167,CRSANR5L15S2019O1N024,NaN,NaN,20181018150000000,jeudi 18 octobre 2018,2,24,AN,15,Session ordinaire 2018-2019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-10-18 15:00:00,None,M. Laurent Saint-Martin
13365,CRSANR5L15S2019O1N030,NaN,NaN,20181022213000000,lundi 22 octobre 2018,2,30,AN,15,Session ordinaire 2018-2019,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-10-22 21:30:00,None,M. Jean-Paul Lecoq
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
663697,CRSANR5L16S2024O1N078,RUANR5L16S2024IDS27903,SCR5A2024O1,20231207090000000,jeudi 07 décembre 2023,1,78,AN,16,Session ordinaire 2023-2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-12-07 09:00:00,None,Un député du groupe LFI-NUPES
663701,CRSANR5L16S2024O1N078,RUANR5L16S2024IDS27903,SCR5A2024O1,20231207090000000,jeudi 07 décembre 2023,1,78,AN,16,Session ordinaire 2023-2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-12-07 09:00:00,None,Un député du groupe RN
663708,CRSANR5L16S2024O1N078,RUANR5L16S2024IDS27903,SCR5A2024O1,20231207090000000,jeudi 07 décembre 2023,1,78,AN,16,Session ordinaire 2023-2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-12-07 09:00:00,None,Plusieurs députés du groupe LR
663726,CRSANR5L16S2024O1N078,RUANR5L16S2024IDS27903,SCR5A2024O1,20231207090000000,jeudi 07 décembre 2023,1,78,AN,16,Session ordinaire 2023-2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-12-07 09:00:00,None,Plusieurs députés du groupe LR
